In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.exposure import match_histograms
from tqdm import tqdm

from dataset import CT2DSliceDataset
from model import DenoisingUNet2D_HybridFiLM


def visualize_prediction(ldct, noise_map, matched_output, ndct, index, out_dir, psnr_val, ssim_val, mse_val, region_maps=None):
    ldct = ldct.squeeze().cpu().numpy()
    noise_map = noise_map.squeeze().cpu().numpy()
    matched_output = matched_output.squeeze()
    ndct = ndct.squeeze().cpu().numpy()

    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    axs[0].imshow(np.clip(ldct, 0, 1), cmap='gray')
    axs[0].set_title("LDCT Input")

    axs[1].imshow(np.clip(noise_map, 0, 1), cmap='hot')
    axs[1].set_title("Noise Map")

    axs[2].imshow(np.clip(matched_output, 0, 1), cmap='gray')
    axs[2].set_title(
        f"Denoised Output\nPSNR: {psnr_val:.2f} dB\nSSIM: {ssim_val:.3f}\nMSE: {mse_val:.4f}"
    )

    axs[3].imshow(np.clip(ndct, 0, 1), cmap='gray')
    axs[3].set_title("NDCT Ground Truth")

    for ax in axs:
        ax.axis('off')

    # Optionally save region overlays if region maps are given
    if region_maps is not None:
        try:
            from matplotlib import cm
            region_maps_np = region_maps.squeeze().cpu().numpy()  # (R,H,W)
            R = region_maps_np.shape[0]
            for r in range(min(R, 3)):  # visualize up to 3 regions as overlays
                fig_reg, ax_reg = plt.subplots(1, 1, figsize=(5, 5))
                ax_reg.imshow(ldct, cmap='gray')
                rm = region_maps_np[r]
                disp = (rm - rm.min()) / (rm.max() - rm.min() + 1e-8)
                ax_reg.imshow(disp, cmap='magma', alpha=0.5)
                ax_reg.set_title(f"Region Map {r+1}")
                ax_reg.axis('off')
                fig_reg.tight_layout()
                fig_reg.savefig(os.path.join(out_dir, f"sample_{index}_region_{r+1}.png"))
                plt.close(fig_reg)
        except Exception as e:
            print(f"Could not save region overlays: {e}")

    os.makedirs(out_dir, exist_ok=True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"sample_{index}.png"))
    plt.close()


if __name__ == "__main__":
    # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    device = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using device: {device}")

    test_data_dir = r"testing 1mm"
    checkpoint_path = r"checkpoint.pt"
    visualization_dir = r"test_visualizations"

    test_dataset = CT2DSliceDataset(test_data_dir, patch_pad=32)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, pin_memory=True, num_workers=4)

    # Important: specify num_regions same as training (3 or appropriate)
    model = DenoisingUNet2D_HybridFiLM(num_regions=3).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    psnr_vals = []
    ssim_vals = []
    mse_vals = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc="Testing")):
            # Handle batches with or without region maps
            if len(batch) == 4:
                ldct, noise_map, ndct, region_maps = batch
                region_maps = region_maps.to(device)
            else:
                ldct, noise_map, ndct = batch
                region_maps = None

            ldct = ldct.to(device)
            noise_map = noise_map.to(device)
            ndct = ndct.to(device)

            output = model(ldct, noise_map=noise_map, region_maps=region_maps)
            output = torch.clamp(output, 0.0, 1.0)

            for i in range(ldct.size(0)):
                pred = output[i, 0].cpu().numpy()
                target = ndct[i, 0].cpu().numpy()

                if np.isnan(pred).any() or np.isnan(target).any():
                    continue

                matched_pred = match_histograms(pred, target, channel_axis=None)
                matched_pred = np.clip(matched_pred, 0.0, 1.0)
                target = np.clip(target, 0.0, 1.0)

                psnr_val = psnr(target, matched_pred, data_range=1.0)
                ssim_val = ssim(target, matched_pred, data_range=1.0)
                mse_val = np.mean((target - matched_pred) ** 2)

                psnr_vals.append(psnr_val)
                ssim_vals.append(ssim_val)
                mse_vals.append(mse_val)

                # Save visualization for first batch, first few samples
                if batch_idx == 0 and i < 3:
                    visualize_prediction(
                        ldct[i], noise_map[i], matched_pred, ndct[i],
                        index=f"b{batch_idx}_s{i}",
                        out_dir=visualization_dir,
                        psnr_val=psnr_val,
                        ssim_val=ssim_val,
                        mse_val=mse_val,
                        region_maps=region_maps[i] if region_maps is not None else None
                    )

    if psnr_vals and ssim_vals and mse_vals:
        print(f"\n✅ [Testing Results] "
              f"Average PSNR: {np.mean(psnr_vals):.2f} dB | "
              f"Average SSIM: {np.mean(ssim_vals):.4f} | "
              f"Average MSE: {np.mean(mse_vals):.6f}")
    else:
        print("\n❌ No valid metrics computed.")
